# 一小时读懂 Data：从供应商原件，到可重现的研究输入

**先看一个数：688981.SH 在 2025-06-10 的成交量。** 原件写 `254439.12`，单位是手；规范记录写 `25443912`，单位是股。Data 保存两者及转换依据。研究程序不用猜单位，也不能把后来收到的财报当作当年已经知道的内容。

本次交付的本地执行版已有保存输出，可以直接阅读；Git 中本文件是源码版，不含读取出的数据表。默认重跑只读取固定历史样本、三张正式版本清单和库存证据，不采集、不构建、不更新正式引用。读取失败保留原异常；固定样本不在本机时，保存输出仍可学习，重新执行会明确报缺样本。

| 阅读段落 | 约用时 | 读完应能回答 |
|---|---:|---|
| 一、存储与 18 个域 | 15 分钟 | 一行行情和一组股东报告各保存什么？|
| 二、版本、修订与 PIT | 15 分钟 | 2026 年才收到的 2025 年报，在 2025 年能用吗？|
| 三、日更、扩展、失败与检查 | 15 分钟 | 明天接上、去年重述、进程失败分别产生什么？|
| 四、View 与消费者交接 | 15 分钟 | Feature、Label、UI 分别拿到什么、保证到哪里？|

时间包含观察输出、思考边界题和对照答案，并非运行耗时。**课程基于产品代码 `c3b75a8ac1aeed0c636ae205d2729670b42868ad`。完整最终测试、最终五类 View 端到端验收、bulk 时间与峰值空间估计仍为 NOT RUN / 未证实；本课程不能据此批准 bulk。**

In [ ]:
from pathlib import Path
import os, sys, json, inspect, platform
import pandas as pd
from IPython.display import display
REPO = next((p for p in (Path.cwd(), Path.cwd().parent, Path.cwd()/"axiom-data")
             if (p/"src/axiom_data/__init__.py").is_file()), None)
if REPO is None:
    raise RuntimeError("请从 axiom-data 仓库根或 notebooks 目录执行；也可先读保存输出。")
sys.path.insert(0, str(REPO/"src"))
from axiom_data import SnapshotReader, FactView, load_raw_batch, load_adjusted_price_view, ArtifactError
from axiom_data.operations import compare_event_projection
from axiom_data.contracts import writable_contracts
REFERENCE_PATH = REPO/"deprecated/data_learning/fixture.json"
INVENTORY_PATH = REPO/"deprecated/data_learning/version_inventory.json"
if not REFERENCE_PATH.is_file() or not INVENTORY_PATH.is_file():
    raise FileNotFoundError("FIXTURE UNAVAILABLE：源码版没有本地数据引用，请按 deprecated/data_learning/README.md 配置；也可阅读已交付的本地执行版。")
REFERENCES = json.loads(REFERENCE_PATH.read_text())
DATA_ROOT = Path(os.environ.get("AXIOM_DEMO_DATA_ROOT", REFERENCES["data_root"]))
SNAPSHOT_ID = REFERENCES["snapshot_id"]
if not (DATA_ROOT/"snapshots"/SNAPSHOT_ID/"manifest.json").is_file():
    raise FileNotFoundError("FIXTURE UNAVAILABLE：固定历史样本缺失，请先读保存输出；明确搬迁后可设 AXIOM_DEMO_DATA_ROOT。")
SYMBOL, START, END = "688981.SH", "2025-06-10", "2025-06-13"
CUTOFF = "2025-06-13T23:59:59+08:00"
reader = SnapshotReader(DATA_ROOT, SNAPSHOT_ID)
VIEWS = REFERENCES["views"]
pd.set_option("display.max_colwidth", 100)
print("固定教学样本：中芯国际 / 4 个交易日 / 18 域；公共只读入口已打开。")
print("Python", platform.python_version(), "| pandas", pd.__version__)
print("历史路径与原身份：deprecated/data_learning/fixture.json；每个业务标签均有对应原引用。")

## 一、存储与 18 个域（约 15 分钟）

### 1. 用户看到一张表，磁盘保存的是可追溯对象

`证券 × 日期 × 字段` 适合行情结果：输入证券、四天区间，输出四行收盘价和成交量。财报的经济时间却是报告期，公布和修订有各自的可知时间；前十大股东是一组报告，缺一个成员会影响整组完整性；行业归属有生效区间。把它们一律铺成“当天数字”，会丢失能否在当时使用的依据。

| 层 | 本例输入 / 输出 | 谁读取 | 磁盘事实 |
|---|---|---|---|
| Raw 原件 | 一次明确请求 → 原响应与观察时间 | 规范构建器、复核人员 | `raw/batches/<批次>/manifest.json` 与 `payload.bin`；不改原件 |
| Canonical 规范记录 | 原字段、来源合同 → 统一键、单位、时间、缺失原因 | SnapshotReader / View 构建器 | 当前数据对象为 `rows.json`，不是 Parquet |
| DomainCommit 域版本 | 一个域完整的分区引用清单、来源和构建配置 | Snapshot / 域依赖 | `canonical/<域>/commits/<版本>/manifest.json` |
| Snapshot 组合清单 | 18 个域版本引用 → 本次固定事实组合 | 查询与 View | `snapshots/<身份>/manifest.json`，不复制 18 域正文 |
| View 消费产物 | 固定 Snapshot、范围、字段、时间/价格口径 → 可重复读取结果 | Research、回放、Qlib、UI 适配层 | 派生目录或 Qlib 导出；必须说明输入事实 |

正式分区对象是 `canonical/<域>/objects/<对象>/rows.json`；日频主要按月，财务按接口与报告年，证券身份按交易所，部分行业成员按证券。历史教学样本还使用域版本内的平铺 `rows.json`，由现有历史读取支持。下方同时展示固定样本与当前正式清单的物理事实，不把老样本布局冒充当前唯一布局。

一个新交易日可能使当月分区变化；新域版本引用新对象并复用其他月份对象。**逻辑上追加一天，不意味着每天复制整库，也不意味着在旧文件尾部直接写入。**

In [ ]:
raw_ref = REFERENCES["examples"]["market_daily"]
raw_batch = load_raw_batch(DATA_ROOT, raw_ref["raw_batch_id"])
raw_row = json.loads(raw_batch.payload)[raw_ref["raw_row_index"]]
display(pd.DataFrame([{k:raw_row[k] for k in ("ts_code","trade_date","close","vol","amount")}]))
prices = reader.market_daily([SYMBOL], START, END)
display(pd.DataFrame(prices)[["symbol","session","close","volume_shares","amount_cny"]])
assert prices[0]["volume_shares"] == 25443912
assert prices[0]["amount_cny"] == 2108011021.0
print("Raw 的 vol：手；amount：千元。公共读取返回股与元，Notebook 未重算正式数据。")
formal_inventory = json.loads((REPO/"deprecated/data_learning/version_inventory.json").read_text())
formal_root = Path(REFERENCES["formal_root"])
formal_snapshots = formal_inventory["formal_inventory"]["snapshots"]
formal_manifest = json.loads((formal_root/"snapshots"/formal_snapshots[0]["snapshot_id"]/"manifest.json").read_text())
market_id = formal_manifest["domain_refs"]["market_daily"]["domain_commit_id"]
market_manifest = json.loads((formal_root/"canonical/market_daily/commits"/market_id/"manifest.json").read_text())
part = market_manifest["partitions"][0]
display(pd.DataFrame([
    {"位置":"历史样本域版本", "布局":"平铺", "对象文件":reader.commits["market_daily"].manifest["output_files"][0]["path"]},
    {"位置":"正式行情域版本", "布局":market_manifest["partition_policy"], "对象文件":"objects/<对象>/rows.json"},
    {"位置":"正式首个分区", "布局":part["key"], "对象文件":f"rows.json / {part['rows']} 行 / {part['bytes']} 逻辑字节"},
]))
print("仅读取正式 manifest，未打开其全历史行情正文；布局证据不是正式闭包验收。")

### 2. 逐域看：这 18 类数据分别解决什么问题？

下面所有样值来自固定原件和对应 Canonical 行。显示内容是投影，不是另一套规范映射；准确 RawBatch 身份、行索引和域版本在历史引用文件中按域名可追查。某域没有 688981.SH 样本时，会显示实际存在的证券，不能拿别人的记录冒充中芯国际。

先分清三个容易混用的量：成交量是**一天交易多少股**，股本是**当时存在多少股**，融资余额是**当日结束欠多少资金**；它们都很大，但不能相互替代。

In [ ]:
# 仅挑展示列；单位/时间与业务含义在下文解释。
FIELD_SPECS = {'security_master': ('身份', 'list_date', 'list_session', '日期'), 'trading_calendar': ('交易日历', 'is_open', 'is_open', '布尔'), 'market_daily': ('行情', 'vol', 'volume_shares', '股'), 'security_status': ('交易状态', 'vol', 'status', '枚举'), 'price_limits': ('限价', 'up_limit', 'upper_limit', '元/股'), 'corporate_actions': ('公司行动', 'cash_div_tax', 'cash_per_share', '元/股'), 'adjustment_factors': ('复权因子', 'adj_factor', 'factor', '无量纲'), 'benchmark_daily': ('基准行情', 'close', 'close', '点'), 'security_capital': ('股本', 'total_share', 'total_shares', '股'), 'financial_events': ('财务事实', 'revenue', 'values.revenue', '元，累计'), 'valuation_daily': ('估值', 'pe', 'values.pe', '倍'), 'universe_membership': ('股票池成员', 'con_code', 'symbol', '证券身份'), 'industry_membership': ('行业成员', 'industry', 'industry_id', '分类标签'), 'holder_count_events': ('股东人数', 'holder_num', 'values.number', '户'), 'top_holders_reports': ('前十大股东', 'hold_amount', 'holders.shares', '股'), 'margin_daily': ('两融', 'rzye', 'values.balance', '元'), 'moneyflow_daily': ('资金流', 'net_mf_amount', 'values.net', '元'), 'forecast_observations': ('业绩预告', 'type', 'values.type', '枚举')}
EXAMPLES = {}
for domain, ref in REFERENCES["examples"].items():
    canonical = reader.commits[domain].rows[ref["canonical_row_index"]]
    raw = load_raw_batch(DATA_ROOT, ref["raw_batch_id"])
    source = json.loads(raw.payload)[ref["raw_row_index"]]
    label, raw_field, canonical_field, unit = FIELD_SPECS[domain]
    if canonical_field == "holders.shares":
        holder = next(h for h in canonical["holders"] if h["name"] == source["holder_name"])
        value = holder["shares"]
    elif canonical_field.startswith("values."):
        value = canonical["values"][canonical_field.split(".")[1]]
    else:
        value = canonical[canonical_field]
    EXAMPLES[domain] = {"域 / 原件标签":domain,"来源接口":raw.manifest["request"]["endpoint"],
        "Raw 字段和值":f"{raw_field} = {source[raw_field]}",
        "规范字段和值 / 单位":f"{canonical_field} = {value} / {unit}",
        "证券或市场":canonical.get("symbol",canonical.get("benchmark",canonical.get("exchange"))),
        "经济时间":(canonical.get("session") or canonical.get("report_period") or canonical.get("effective_from") or canonical.get("effective_date") or canonical.get("list_session"))}
assert set(EXAMPLES) == set(writable_contracts()["domains"])
print("当前可写合同域清单：18 域；历史样本覆盖同一组域，但合同版本可能较早。")

#### 身份、日历与市场：看清“能交易”“成交了多少”“最高允许什么价”

1. **security_master（证券身份）**：`stock_basic.list_date → list_session`，键是证券身份；上市起点和退市边界描述身份有效区间，不自动证明每天有成交。未退市时终点为空属于正常开放区间，不能填成某个未来日期。正式证券身份的边界 authority 按 Design 来自交易所；当前正式 manifest 绑定 exchange_security.v1。这里 stock_basic 只是历史教学样本的实际原件，不能代表正式身份 authority，也不证明真实退市冲突已解决。
2. **trading_calendar（日历）**：`trade_cal.is_open → is_open`，键是交易所与日；这是市场是否开市，不是某股是否停牌。周末 false 正常；请求覆盖之外没有行需另查覆盖，不能当作休市。
3. **market_daily（行情）**：`daily.vol → volume_shares`，手转股；`amount` 千元转元。键是证券与交易日；OHLC 是真实未复权价格。上市前不适用，正常应交易日缺行不应补零。行情里有汇总状态字段，不替代状态、限价域各自的来源解释。
4. **security_status（交易状态）**：本例用 `daily` 实际成交观察推得 `normal_active`；显式停复牌证据也参与正式解释。键是证券与日。找不到行情不能凭空推得 suspended，缺口和有证据停牌不同。
5. **price_limits（涨跌停价）**：`stk_limit.up_limit/down_limit → upper_limit/lower_limit`，单位元/股，键是证券与日。告诉回放交易约束，不是当天最高最低成交价；无限价需显式状态，缺限价证据不能猜规则。
6. **corporate_actions（公司行动）**：`dividend.cash_div_tax → cash_per_share`，元/股；按行动身份、证券、有效日及版本保存，还保留公告、登记、除权、支付日。下方样本是 000001.SZ 的分红。没有分红不等于漏抓，尚在预案阶段的空除权日也不能当已实施；必须看行动状态与请求覆盖。

In [ ]:
display(pd.DataFrame([EXAMPLES[d] for d in ("security_master","trading_calendar","market_daily","security_status","price_limits","corporate_actions")]))

#### 稳定市场尺度与财务：原始数字、换算口径、预测特征要分清

7. **adjustment_factors（复权因子）**：`adj_factor.adj_factor → factor`，无量纲，键是证券与日。用于价格口径转换，不是分红现金，也不是成交价格；指定锚点日无因子需要显式规则和来源，不准用 1 补齐。
8. **benchmark_daily（基准行情）**：`index_daily.close → close`，指数点，键是基准身份与日。这里是 000300.SH；不能当成一只股票的元/股，也不能用股票日历存在推得基准记录齐全。
9. **security_capital（股本）**：`daily_basic.total_share → total_shares`，来源万股转股，键是证券与日。总股本、流通股本是存量；它们用于市值与比例，和 daily 成交量不同。来源未提供时保留缺失，不把上一日无条件延用成新事实。
10. **financial_events（财务事实）**：`income.revenue → values.revenue`，元。收入/现金流累计到报告期，资产负债表是期末存量；同域还包含 balancesheet、cashflow、fina_indicator。键需证券、接口、报告期、报表类型、修订；公告/观察时间另存。下方是 2024 三季累计收入，不能直接当作第三季度收入。缺列与尚无可见报告不同。
11. **valuation_daily（估值）**：`daily_basic.pe → values.pe`，倍；键是证券、日和修订。供应商 PE 并不自动等于 Research 自己定义的盈利口径；亏损、分母缺失可能为空，不能把空 PE 当 0 倍便宜。
12. **universe_membership（股票池成员）**：`index_weight.con_code → symbol`，键是组、证券、生效区间和修订；权重观察形成完整组状态。区间常用左闭右开，`effective_to` 当日已不属于这个区间。终点为空可表示开放；没有一条成员行不一定证明不属于，需完整组和覆盖证据。

In [ ]:
display(pd.DataFrame([EXAMPLES[d] for d in ("adjustment_factors","benchmark_daily","security_capital","financial_events","valuation_daily","universe_membership")]))

#### 分类、持有人与资金：组完整性和时点不能省略

13. **industry_membership（行业成员）**：`bak_basic.industry → industry_id`，本例为半导体；键是分类组、证券、生效区间、修订。供应商分类与其他分类体系不能混成同一标签；本例来自历史 bak_basic 分类；当前正式行业合同 v3 使用 SW2021 分类体系及其历史成员证据，两种分类不能混用。本例只是某日观察区间，不证明全年一直属于该行业。未分类、正常不适用、未知漏抓应区分。
14. **holder_count_events（股东人数）**：`stk_holdernumber.holder_num → values.number`，户；键是证券、报告期和修订，另存公告时间。不是每天测一次的持有人数；报告期到下一次披露间能否用于某天，由 PIT 和叶事实读取规则解释。报告日期缺失会使原件无法形成规范事件，不得杜撰日期。
15. **top_holders_reports（前十大股东）**：`top10_holders.hold_amount → holders[].shares`，股，`hold_ratio` 为百分比。下面显示报告中一个真实成员；整个报告按证券、报告期、修订保存十名成员及组完整性，名字映射到持有人身份。只抓到 9 名时不能把九人合计冒充前十合计；来源冲突需显式缺失/资格。
16. **margin_daily（两融）**：`margin_detail.rzye → values.balance`，元；键是证券、经济日和修订。`rzmre` 是当天融资买入流量，`rzye` 是余额；二者不是同一个指标。融券量单位为股；负还款等异常来源按合同资格保留问题，不能静默取绝对值。
17. **moneyflow_daily（资金流）**：`moneyflow.net_mf_amount → values.net`，来源万元转元；键是证券、日、修订。本例负值表示净流出，不是缺失；来源按成交分类的资金流不等于会计现金流。缺接口响应要调查覆盖，不能填成零流入。
18. **forecast_observations（业绩预告）**：`forecast.type → values.type`，枚举；键是证券、报告期、修订，另存公告与观察时间。下方是 000401.SZ 的“续亏”；不是已审计利润，且不是所有公司每期必须发布。无预告能否视为正常取决于需求和来源覆盖，不能拿它代替已披露财报。

In [ ]:
display(pd.DataFrame([EXAMPLES[d] for d in ("industry_membership","holder_count_events","top_holders_reports","margin_daily","moneyflow_daily","forecast_observations")]))

**第一段自测（先想再展开）**：某股 6 月 10 日有 2,544 万股成交，股本约 79.9 亿股；这两列能合并吗？财报里的三季累计收入，能直接给 7—9 月收入 Feature 吗？日历开市而某股无行情，能标成停牌吗？

<details><summary>对照答案</summary>

不能合并：成交量是日流量，股本是时点存量。三季累计收入需按正确报表类型减去半年累计，且两个输入都满足 cutoff；本课程不重写派生算法。开市只能证明市场开放，证券无行情需要停牌、身份或来源覆盖证据；未知缺口不能变成正常状态。

</details>

## 二、版本、修订与 PIT（约 15 分钟）

### 3. 三个 Snapshot，不是三份全库

库存盘点时间为 **2026-09-26T13:22:17Z**：3 个 Snapshot、21 个 DomainCommit、2,121 个 View、299,229 个 RawBatch。三个 Snapshot 各引用 18 域，共享 17 个域版本，仅财务分别为 v2/v3/v4；财务有四个域版本目录，并不等于每个都被这三张清单选用。

Raw 清单声明 257,783 种内容摘要：采集批次数、内容种类、磁盘副本数是三种问题。此盘点没有重新读/hash 全部 payload，也没有重做闭包验证。下方只重新读取三张正式 manifest，核对组合共享事实，保留旧清单原样。

**三个版本轴**：合同版本说明字段/键/单位/时间语义；代码版本说明由什么实现生成；数据身份说明哪组不可变内容。改 Notebook 文字不使财报合同自动升级；财务合同变化也不等于行情数值变化。当前没有盘点到正式 `current` 指针，不能按创建时间叫某张“生产最新版”。

In [ ]:
compositions = []
for item in sorted(formal_snapshots, key=lambda x:x["created_at"]):
    manifest = json.loads((formal_root/"snapshots"/item["snapshot_id"]/"manifest.json").read_text())
    compositions.append(manifest["domain_refs"])
    display(pd.DataFrame([{"Snapshot":item["snapshot_id"], "财务合同":manifest["domain_refs"]["financial_events"]["contract_version"],
        "域数":len(manifest["domain_refs"]), "创建时间":manifest["created_at"], "清单记录状态":manifest["validation_summary"]["status"]}]))
shared = [d for d in compositions[0] if len({c[d]["domain_commit_id"] for c in compositions})==1]
changed = sorted(set(compositions[0])-set(shared))
print("共同引用",len(shared),"域；变化域",changed)
assert len(shared)==17 and changed==["financial_events"]
print("旧清单仍存在且组合未改；未执行旧版本全历史查询或闭包重验。")

### 4. 一份报告为什么要保存两种时间和多次修订？

财务物理记录不仅有 `symbol/report_period/values`：还保留接口、报表类型、逻辑事件键、修订身份、原件引用、首次观察、供应商日期和观察序列。相同数值再次收到可以追加观察而不制造新经济修订；收入从 100 改为 90 才是内容变化（这两个数是**示意**，不是本样本测值）。

供应商标注 2025-04-30 披露，不代表本系统 2025-04-30 持有这一修订。该样本在 2026 年实际收到；下方对同一 `2025-06-13` cutoff 比较“按供应商日期探索历史”与“本系统当时实际持有”。空的 operational 结果说明观察证据不足，不说明公司没有报告。

| policy：选择时间的规则 | 当前实现 | 该样本历史使用含义 |
|---|---|---|
| `market_pit_safe_v1` | 支持；没有修订公开证据时使用实际观察时间 | 保守处理；名字不把 best_effort 自动升级为严格 |
| `operational_pit_v1` | 支持 | 本系统实际观察后才可见 |
| `best_effort_vendor_v1` | 支持 | 供应商日期用于非严格历史探索，资格仍保留 |
| bootstrap_hybrid | 设计保留，未实现 | 不可把此名字传入当作现成功能 |

**qualification 是另一轴**：`observed` 有实际观察；`best_effort` 历史可知证据较弱；`unknown` 不形成可见版本；`verified` 当前明确拒绝 `VERIFIED_EVIDENCE_UNAVAILABLE`。一列 ann_date 不能证明某次具体修订早已公开，也不能升级成 verified。

In [ ]:
financial = reader.as_of("financial_events", symbols=[SYMBOL], knowledge_cutoff=CUTOFF, pit_policy="best_effort_vendor_v1")
observed = reader.as_of("financial_events", symbols=[SYMBOL], knowledge_cutoff=CUTOFF, pit_policy="operational_pit_v1")
safe = reader.as_of("financial_events", symbols=[SYMBOL], knowledge_cutoff=CUTOFF, pit_policy="market_pit_safe_v1")
display(pd.DataFrame([{"问题":"供应商日期历史探索","可见记录":len(financial)},
                     {"问题":"本系统实际持有","可见记录":len(observed)},
                     {"问题":"无 verified 证据的 safe","可见记录":len(safe)}]))
display(pd.DataFrame(financial)[["symbol","endpoint","report_period","report_type","vendor_available_at","first_observed_at","pit_qualification"]].head(4))
assert financial and not observed and not safe
example_event = financial[0]
display({"证券":example_event["symbol"], "接口":example_event["endpoint"], "报告期":example_event["report_period"],
         "逻辑事件键":example_event["logical_event_key"], "修订身份":example_event["revision_id"],
         "值":example_event["values"], "观察次数":len(example_event.get("observations",[example_event])),
         "原件追查":"deprecated/data_learning/fixture.json 的 financial_events 原件标签；其他返回行在公共读取结果保留 source_ref"})

### 5. TTM 是稳定事实派生；质量打分是研究配方

举一个**算式示意**：同口径 2025 半年累计收入 60、2024 全年 100、2024 半年 40，TTM 收入为 `60 + 100 - 40 = 120`。如果前期半年缺失，120 无法成立；若 2024 全年后来重述为 90，则未来 cutoff 的结果应按可见修订改变，旧 cutoff 不应提前看到 90。资产总额属于期末存量，不能套收入累计差分。

真实调用 `SnapshotReader.financial_derived(...)` 复用正式派生选择与缺期规则，返回单季/TTM/基础比率及组成和缺失原因。下方只展示实际结果字段；不把上面的 120 写进事实样本。20 日动量、质量分及横截面标准化由 Research 定义，正式 Feature 执行应走共同 Engine/Core 路径，课程不新增另一套计算。

In [ ]:
derived = reader.financial_derived(symbols=[SYMBOL], knowledge_cutoff=CUTOFF, pit_policy="best_effort_vendor_v1")
print("实际稳定派生记录数",len(derived))
if derived:
    columns = [k for k in ("symbol","report_period","field","value","missing_reason","pit_qualification") if k in derived[0]]
    display(pd.DataFrame(derived)[columns].head(8))
    print("返回结构中的组成与来源字段：",[k for k in derived[0] if "component" in k or "source" in k or "reason" in k])
else:
    print("该固定样本没有符合此 cutoff 的派生结果；不编造数值。")

**第二段自测**：一份报告经济期为 2024 年、供应商公告日期在 2025 年、系统首次观察在 2026 年。2025 年决策的 Feature 可用它吗？若 best_effort 能返回，是否代表 strict PIT 通过？如果只改了财务合同，另外 17 域需要复制吗？

<details><summary>对照答案</summary>

operational / 当前 safe 在 2025 年不可见。best_effort 可用于明确非严格的历史探索，但不能声明系统当年拥有内容。三个时间回答不同问题。新 Snapshot 可以引用新财务版本和既有 17 域；是否需重建某类 View 还要看它绑定的 Snapshot、依赖与执行计划，不能用域复用推导所有 View 免费复用。

</details>

## 三、日更、扩展、失败与检查（约 15 分钟）

### 6. 明天接上：固定父版本 → 保存观察 → 受影响分区 → 候选组合

假设今天已有周一至周四（**过程示意，未在本 Notebook 发起 daily**）。调用方显式给出父 Snapshot、run_id、每个来源的经济窗口/可用性策略、必要固定 Raw 和域配置。周五行情和周四两融可以属于同一计划，但不能把 T+1 接口当作周五已经齐全。`plan_daily` 仅检查计划，不证明来源已有数据。

| 场景 / 输入 | Raw 和域版本发生什么 | Snapshot / 可读性 | 已有实现、此次证据 |
|---|---|---|---|
| 新增周五记录 | 保存新观察；变化月份另写对象，其余分区复用；新域版本列完整引用 | 候选组合只替换变化域；旧 Snapshot 原值保留 | `daily`；本课程未执行，最终 E2E NOT RUN |
| 相同观察重放 | 显式 `observed_raw_batch_ids` 复用同一已存观察，不重取、不造新观察时刻；canonical 确无变化时 NO_CHANGE | 返回原 Snapshot 身份，不生成无意义新组合 | `daily`；本课程未执行 |
| 去年财报修订 | 新原件和经济修订；财务受影响的报告年分区变化；依赖影响不能只检查今天 | 新财务组合；旧组合仍解释原报告 | `daily` / `repair`；本课程未执行 |
| 采集/构建中断后 resume | 成功 Raw、完成产物和冻结计划保留；同 run_id 校验已完成引用后续跑 | 失败不把候选当正式；父版本继续供读 | 同计划恢复；最终 E2E NOT RUN |

重复请求供应商得到相同 payload **不一定**就是 NO_CHANGE：新观察时间可能影响观察序列/证据合同。要区别“重放同一观察”和“再收到一次同样内容”。任一必需来源失败，会在候选构建前停止并保留 Raw；构建失败也不允许移动正式引用。

目前 `daily` 成功返回 `CANDIDATE_BUILT` 或 `NO_CHANGE`，下一阶段是 `REQUIRED_VIEWS_AND_FULL_ADMISSION`，`ready_for_consumption=false`。得到候选不是生产日更验收完成。单 run 锁拒绝同身份并发；恢复验证冻结计划，换输入必须新 run_id。

### 7. 生产安全保护实际保什么？

下面是已存在入口的调用形状，**说明用、不执行**：

```python
plan_daily(DATA_ROOT, explicit_parent_snapshot, source_requests=requests)
daily(DATA_ROOT, run_id=explicit_run_id, snapshot_id=explicit_parent_snapshot,
      source_requests=requests, domain_inputs=frozen_inputs,
      observed_raw_batch_ids=existing_observations)
repair(DATA_ROOT, run_id=explicit_repair_run, snapshot_id=explicit_parent_snapshot,
       domain_inputs=frozen_inputs)
materialize_views(DATA_ROOT, run_id=explicit_view_run,
                  snapshot_id=explicit_candidate, views=explicit_view_descriptors)
```

这些是写操作，参数值需来自正式计划。本课程执行单元不调用它们。

- **冻结代码与输入**：内置构建器/五类 View 捕获实际 Python、合同资源及运行时引用，并执行捕获包；后续修改工作区不会替换同 run 的已捕获实现。它不打包所有外部依赖，不替采集器伪造历史观察。
- **失败不发布**：Raw 成功、域构建成功、候选成功、Views 成功、admission 通过是不同状态。必要输入/验证失败，不能叫 ready，也不能仅看进程退出码。
- **断网恢复**：备份应包含原件、Canonical、父链/依赖、Snapshot、所需 View、代码包和调用记录。`verify_recovery` 验证已恢复 root；它不代替备份复制。离线 guard 禁止恢复过程中补拉供应商数据。
- **中断续跑**：操作记录写明已完成阶段、原引用与计划；resume 加载并验证这些对象。完成标记不允许跳过对象真实性检查。备份少了某个代码包或对象，即使目录看似齐全也会失败。

有关实际操作请读 [daily](../docs/operations/daily.md)、[repair](../docs/operations/repair.md)、[恢复](../docs/operations/recovery.md) 与 [冻结执行](../docs/operations/frozen-execution.md)。当前课程保存输出只证明读取窗口；上述完整生产行为的最终测试/恢复演练未在本次课程运行。

### 8. 供应商多了一个字段，要改哪里？

**例 A：Raw 中已经有新字段。** 假设以后需要 `daily_basic` 的另一项已保存在原件的字段（设计示意，未承诺当前可读）。先确认含义、原单位、规范单位、证券/日期/修订键、可知时间、正常空值与来源。再修改相应来源映射和版本化合同，注册新的可写合同版本，通过现有 `BuildApplication` / `repair` 的共享构建路径用明确 Raw IDs 构建新 lineage，得到新 DomainCommit 与 Snapshot。旧合同留作只读解释，不往旧合同父链混写新语义；公共读取 schema、字段说明和定向验收一并更新。消费者明确请求新 Snapshot/字段，旧版本仍能按原合同读。

**例 B：Raw 根本没有字段。** 先在来源配置/明确计划中授权采集并保存新原件，再按同一路径构建。不能从网上现在的值补进旧 Raw，冒充当时持有。只有供应商多给一个数，也不能跳过单位、日期或修订规则。

**例 C：稳定派生。** 用现有财报叶事实扩展清晰会计比率，说明分母、累计/单季口径、缺期和重述，沿 `financial_derived` 与既有财务 View 职责实现；新增协议由独立 ticket 裁决，不在 Notebook 写正式公式实现。

**例 D：研究 Feature。** 20 日动量要固定 Data 引用、决策 cutoff、窗口、缺失和配方，归 Research 的 FeaturePlan / FeatureBuild，执行复用总体设计规定的公共路径。Data 提供可追溯事实，不替 Research 决定预测配方。UI 也不自行重算质量分。

| 扩展对象 | 必须固定的输入 | 新产物 / 读取变化 | 当前可调用或状态 |
|---|---|---|---|
| Raw 新字段进入规范域 | 来源规则、原件、键、单位、时间、空值 | 新合同/DomainCommit/Snapshot；旧合同只读 | 现有构建/repair 路径；此字段示例未实现 |
| TTM/基础比率 | 已准入财报及 cutoff | 稳定派生值、组成和缺失原因 | `SnapshotReader.financial_derived` 已展示 |
| 新预测特征 | 固定事实、FeaturePlan、决策范围 | FeatureBuild 与配方 lineage | 跨仓接入未完成；协议说明 |

没有用本课程创建任何新域、合同或正式产物。

### 9. 写入时检查过，为何读取仍要检查？

例子：构建时四天 View 通过。半年后文件被截断，或调用方把整月请求交给这四天 View；写入时的 PASS 不能证明这次请求可以使用。检查需针对新问题，不是每读一股重放全库原件。

| 检查 | 何时做 / 范围 | 防止什么 | 何时能复用 |
|---|---|---|---|
| 来源与规范语义 | 新构建/新来源闭包准入；所需 Raw、映射、单位、键、修订 | 手当股、修订丢失、来源错误 | 固定输入与合同、代码绑定仍适用时，在操作级共享 |
| Snapshot 和直接引用 | Reader 打开、每次消费边界；所用 manifest/域引用 | 组合错配、检查后换文件 | 对象绑定不变时复用既有元信息核验 |
| 分区/View 内容完整性 | 请求实际消费的对象，包含所读分区的完整顺序/数量/摘要检查 | 文件损坏、月分区后部被截断 | 同一对象及消费状态未改变；不是忽略余下月份内记录 |
| 请求范围/字段/口径 | 每次具体请求 | 四天冒充一月，错字段、错价格/PIT 口径 | 不同请求要按其范围判断；不能复用较窄范围的结论 |
| 任务资格与缺口 | View 准入、Research/UI 交接 | best_effort 冒充 strict、部分结果用于正式研究 | 要求与证据绑定的固定范围匹配 |
| 全历史闭包复核 | 显式 `validate_snapshot_closure` 或需要的新来源闭包 | 未消费历史、父链、Raw/mapping 问题 | 普通读不声称全库已重验；按操作共享必要检查 |

`SnapshotReader.market_daily` 返回请求区间存在的行，**不提供自动整月完整性 PASS**；较高层已物化 View 范围检查与 admission 才能对具体用途给相应保证。不能把“读出四行”当成“需求已完整”。本课程不新增额外通用 gate。

### 10. 空值、空结果与缺行不是一回事

| 例子 | 正确理解 | 不能做什么 |
|---|---|---|
| 来源 row 存在、某字段为 null | 保留 `missing_reasons` 如 vendor_null / not_provided | 补成 0，冒充零利润 |
| report_period 存在，但 cutoff 下无可见报告 | 当前可知范围没有这份报告 | 说公司当年没报告，或读未来修订 |
| 身份生效前、明确未上市 | 该区间不适用 | 伪造价格或误算收益 |
| 显式停牌证据且无成交 | 正常非交易状态，并保留依据 | 无条件按前收盘伪造成交价 |
| 应有行情但来源覆盖/记录不足 | 未知缺口；调查或阻断 | 从“没数据”推断停牌 |
| 股东报告仅九人或有冲突 | 报告组不完整/资格不足，聚合明确缺失 | 把九人合计标成前十大 |

下方寻找固定样本内真实 Canonical 空值并显示原因。找不到的类别明确 FIXTURE UNAVAILABLE；其他几类仍是上表的说明示例，不能伪造一个停牌日凑验收。

In [ ]:
missing_examples=[]
for domain, commit in reader.commits.items():
    for row in commit.rows:
        reasons=row.get("missing_reasons",{})
        if reasons:
            missing_examples.append({"域":domain,"证券":row.get("symbol"),"经济期":row.get("session") or row.get("report_period"),
                                     "真实缺失原因":reasons})
        if len(missing_examples)>=5:break
    if len(missing_examples)>=5:break
if missing_examples:display(pd.DataFrame(missing_examples))
else:print("FIXTURE UNAVAILABLE：固定样本没有带 missing_reasons 的规范空值实例；表格只是规则说明。")
print("真实停牌、九人成组冲突及完整断网恢复：本次课程未执行，不据此记 PASS。")

**第三段自测**：周五来源采集到一半中断，周四父 Snapshot 是否还能读？同 run_id 恢复能改参数补拉“今天最新”吗？添加一个字段需改掉历史对象摘要吗？四天通过的 View 能证明整个六月完整吗？

<details><summary>对照答案</summary>

已发布父版本保持可读；成功 Raw 保留待恢复。恢复固定原输入/配置/代码，同身份不能换计划，不能拿最新供应商数据冒充恢复原观察。新字段产生明确的新合同/lineage 与版本，历史摘要不改。四天证据只覆盖四天，更大请求必须使用覆盖对应范围的产物和证据。

</details>

## 四、View 与消费者交接（约 15 分钟）

### 11. 同一个 Snapshot，两个真实价格口径

原始行情 `FactView.read("market_daily", price_basis="unadjusted")` 与已建复权产物 `read("adjusted_price", price_basis="anchor_adjusted")` 使用同一 Snapshot。前者用于真实价格观察/交易回放；后者固定 `2025-06-13` 锚点，用 `未复权价 × 当日因子 ÷ 锚点因子` 作研究换算。

同样都叫 close，含义不同。本样本因子为 1，所以两列实际相等；相等不是例子失败，它提醒你**不能靠数值猜口径**。若当日因子 2、锚点 4，82 元会换成 41 元（仅算式示意，非真实样本）。不能把 41 元送给成交回放当真实成交价。复权产物的 `research_non_pit` 也不能被当作决策时 strict Feature 输入。

锚点状态不能沿用旧课程“尚待裁决”：当前实现已有明确请求/来源锚点和资格规则；但 600069.SH 的真实官方列表与公告日期冲突，最终不可变纠正/准入证据仍未在本课程通过。旧 Snapshot 的日期不是已核实事实，synthetic 日期几何测试不能替代真实来源验收。

In [ ]:
facts = FactView(DATA_ROOT, SNAPSHOT_ID, adjusted_price_view_id=VIEWS["adjusted"])
original = facts.read("market_daily", symbols=[SYMBOL], start_session=START, end_session=END,
                      fields=["session","symbol","close"], price_basis="unadjusted")
adjusted = facts.read("adjusted_price", symbols=[SYMBOL], start_session=START, end_session=END,
                      fields=["session","symbol","close","adjustment_state"], price_basis="anchor_adjusted", pit_policy="research_non_pit")
left=pd.DataFrame(original["rows"]).rename(columns={"close":"未复权 close"})
right=pd.DataFrame(adjusted["rows"]).rename(columns={"close":"锚点复权 close"})
display(left.merge(right,on=["session","symbol"]))
view = load_adjusted_price_view(DATA_ROOT,VIEWS["adjusted"])
display({"共享 Snapshot":SNAPSHOT_ID,"复权 View":VIEWS["adjusted"],"固定范围":view.manifest["scope"],
         "锚点":view.manifest["anchor_session"],"decision_cutoff":view.manifest["decision_cutoff"],
         "价格口径":view.manifest["price_basis"],"PIT policy":view.manifest["pit_policy"],
         "资格":view.manifest["pit_qualification"],"来源域":list(view.manifest["domain_refs"])})
assert all(a["close"]==b["close"] for a,b in zip(original["rows"],adjusted["rows"]))

### 12. 已有 View 内的子集直接读，越界不偷偷扩建

默认请求使用该已建复权 View 的整个固定范围（两只证券、四天）；显式查中芯国际一天是合法子集。请求整月会得到 `INSUFFICIENT_SCOPE`；不给 View 引用会得到 build-required。读取没有采集或构建权限，用户不应因读图而启动供应商调用。

默认 `fields=None` 使用这个 View 的已有字段，并不是自动发现供应商新字段。`pit_policy="actual"` 返回已绑定口径，也不是“请给最严格口径”。如果要严格资格，必须验证所用产物本来就能满足，不能把参数当作升级按钮。

In [ ]:
default_rows = facts.read("adjusted_price",price_basis="anchor_adjusted")["rows"]
subset_rows = facts.read("adjusted_price",symbols=[SYMBOL],start_session=START,end_session=START,
                         fields=["session","symbol","close"],price_basis="anchor_adjusted")["rows"]
print("默认覆盖行数",len(default_rows),"；单日合法子集行数",len(subset_rows))
display(pd.DataFrame(subset_rows))
refusals=[]
for label, query in [
    ("四天 View 查整月",lambda:facts.read("adjusted_price",symbols=[SYMBOL],start_session="2025-06-01",end_session="2025-06-30",price_basis="anchor_adjusted")),
    ("没有显式 View 引用",lambda:FactView(DATA_ROOT,SNAPSHOT_ID).read("adjusted_price",price_basis="anchor_adjusted")),
    ("请求无法提供的 verified 资格",lambda:facts.read("adjusted_price",price_basis="anchor_adjusted",pit_policy="verified")),
    ("调用未实现的 hybrid policy",lambda:reader.as_of("financial_events",symbols=[SYMBOL],knowledge_cutoff=CUTOFF,pit_policy="bootstrap_hybrid")),
]:
    try:query()
    except (ArtifactError,ValueError) as exc:refusals.append({"请求":label,"真实拒绝":str(exc)})
    else:raise AssertionError("应拒绝的请求成功："+label)
display(pd.DataFrame(refusals))

### 13. 真实公共读取接口，返回保证到哪里？

以下函数已存在。本课程实例固定 root/Snapshot/证券/日期；均不写数据。`SnapshotReader` 返回记录 tuple，记录包含领域的值与来源/时间；`FactView.read` 返回 rows、字段和所用引用、价格/PIT/来源质量等元信息。构造函数推荐业务参数：`FactView(data_root, snapshot_id, *, adjusted_price_view_id=None, financial_fact_view_id=None)`；历史兼容参数不作为新课程用法，原标识在 deprecated 中可查。

| 真实接口 | 用途 / 固定输入 | 越界、缺口与保证边界 |
|---|---|---|
| `SnapshotReader(root, snapshot_id)` | 打开明确事实组合 | 验引用及消费对象；不授予全市场完整性 |
| `.market_daily(symbols, start_session, end_session)` | 未复权实际行情 | 返回存在行；空/少行不是覆盖 PASS |
| `.facts(domain, symbols=..., start_session=..., end_session=...)` | 规范事实与领域元信息 | 按域支持情况拒绝不适用查询；不自动选 PIT 修订 |
| `.as_of(domain, knowledge_cutoff=..., pit_policy=..., symbols=...)` | 在 cutoff 下选择事件/修订 | 未可见返回空；policy 不支持报错；不升级资格 |
| `.financial_derived(knowledge_cutoff=..., pit_policy=..., symbols=...)` | 稳定会计派生 | 缺期/冲突保留原因；不计算预测配方 |
| `.leaf_fact(leaf, symbol=..., target_session=..., knowledge_cutoff=..., pit_policy=...)` | 指定叶事实与资格 | 返回值、可用状态、来源和原因；使用已有覆盖规则 |
| `FactView(...).read(domain, ...)` | 明确事实/已建财务或价格 View | 已物化产物拒绝范围/字段/口径越界；缺引用失败 |
| `QlibViewReader(root, view_id)` | 已导出的日历、数值与元信息 | 数据不适用/缺失按产物协议返回；不是新事实来源 |

下面打印真实方法签名，并打印本次复权返回的键。参数名中的 cutoff 不能脱离具体域/物化策略理解。

In [ ]:
for fn in (SnapshotReader.market_daily,SnapshotReader.facts,SnapshotReader.as_of,
           SnapshotReader.financial_derived,SnapshotReader.leaf_fact,FactView.read):
    print(fn.__qualname__, inspect.signature(fn))
print("FactView 实际返回键：",list(adjusted))
print("行情记录仍需用途准入；只读返回成功不是 Research FeatureBuild 验收。")

### 14. Feature、Label 和 UI：三次交接，三种权限

**Feature 交接（Data 可调用；Research 接入未完成）**：设决策日为 2025-06-10，知识截止当日收盘。读取明确 Snapshot 的历史行情范围，并用 `.as_of("financial_events", knowledge_cutoff=..., pit_policy=...)` 获取截止时可见财报；连同固定引用、字段、缺失状态、资格交给 FeaturePlan。窗口不足或 operational 财报为空时，Feature 不能借未来结果补齐。Data 不知道你的模型需要 20 天窗口还是一年，Research 要把需求与证据对齐。

**Label 交接（Data 可调用；跨仓权限隔离尚未实现）**：若研究明确授权未来三天收盘结果，读取 6/11—6/13 的结果区间作为 Label 输入。标签含未来是任务本身；这个权限必须独立记录，不能把 Label 的未来数据/资格送进 Feature。当前 Data 原始行情接口按显式区间读取，**不会自动替你执行 Feature/Label 的跨仓权限隔离**。下方只展示两份独立 Data 输入，没有创建 FeatureBuild/LabelBuild，更没有训练模型。

**UI 交接（Data 可调用；UI 应用尚未接入）**：UI 画某个固定 View 的曲线，显示 Snapshot、口径、区间与缺口。若请求整月而只有四天，可明确提示缩小区间或标明部分覆盖；不能静默把四天画成整月完整，也不能让部分展示资格流进正式研究。当前 Data 越界错误已经展示；“UI 部分展示授权”是总体设计要求，不是本课程已经实现的 HTTP 服务。

截至核对，Research 现有源码主要是合同/协议层，未找到真实 SnapshotReader/FactView 接入；workspace 没有 UI 仓库。Design 的 `build_features/build_labels`、`POST /charts/query` 是协议示意，不能把它们列为当前已部署 API 或 Data 集成 PASS。

In [ ]:
decision_session="2025-06-10"
decision_cutoff="2025-06-10T23:59:59+08:00"
feature_prices = reader.market_daily([SYMBOL],decision_session,decision_session)
feature_financial = reader.as_of("financial_events",symbols=[SYMBOL],knowledge_cutoff=decision_cutoff,pit_policy="operational_pit_v1")
label_prices = reader.market_daily([SYMBOL],"2025-06-11","2025-06-13")
display(pd.DataFrame([
    {"接收者":"Feature 输入示意","Data 真实查询范围":decision_session,"Data 返回行":len(feature_prices),"知识时间":decision_cutoff,"边界":"只有一天；不满足20日窗口"},
    {"接收者":"Feature 财报输入示意","Data 真实查询范围":"截至决策时可见财报","Data 返回行":len(feature_financial),"知识时间":decision_cutoff,"边界":"无operational可见财报"},
    {"接收者":"Label 输入示意","Data 真实查询范围":"2025-06-11 至 2025-06-13","Data 返回行":len(label_prices),"知识时间":"明确授权未来结果；不用作Feature","边界":"尚未创建LabelBuild"},
    {"接收者":"UI 输入示意","Data 真实查询范围":f"{START} 至 {END}","Data 返回行":len(adjusted["rows"]),"知识时间":view.manifest["pit_policy"],"边界":"显示研究价格口径；无HTTP接入"},
]))
assert all(row["session"]<=decision_session for row in feature_prices)
assert all(row["session"]>decision_session for row in label_prices)
print("两份范围分开只是教学调用检查，不证明跨仓权限控制已经实现。")

### 15. 同一组键和值，导出后还要对得上

Qlib 是数值消费后端，不产生另一套财务或持有人事实。已有 `compare_event_projection` 对固定事件 View 与其数值投影比较证券、日历、字段、缺失状态和数值。下面实际检查一只证券、四天、三字段。

结果 PASS 只覆盖该历史产物和查询。它不证明当前五类 View 全链路验收、不证明全历史/全市场完整、不证明 strict PIT，也不证明 Feature/Label/UI 已接通。原事件 View 持久身份位于 deprecated 引用；这里以业务标签显示该检查对象。

In [ ]:
comparison=compare_event_projection(DATA_ROOT,SNAPSHOT_ID,VIEWS["event"],symbols=[SYMBOL],
    fields=["holder.number","holder.top10_ratio","margin.balance"],start_session=START,end_session=END)
display({"status":comparison["status"],"对象":"固定事件事实View及Qlib投影（原ID见deprecated引用）",
         "rows":comparison["rows"],"fields":comparison["fields"],
         "start_session":comparison["start_session"],"end_session":comparison["end_session"]})
assert comparison["status"]=="PASS"

### 16. 到这里，哪些是已验证，哪些还欠证据？

| 项目 | 本课程状态 | 实际证明 / 限制 |
|---|---|---|
| 环境与固定样本公共读取 | PASS | 本 Notebook 执行，保存真实输出；不等于整个测试 suite |
| 18 域原件 → 规范样值 | PASS | 固定原件与对应行可追查；早期合同示例不替代当前正式域重建 |
| 正式三个组合共享 17 域 | PASS：manifest 比较 | 未读取正式全历史正文或重新校验全部来源闭包 |
| PIT 三策略历史反例 | PASS | 同一 cutoff 的返回差异；verified 不可用，hybrid 未实现 |
| 两种真实价格口径、合法子集与拒绝越界 | PASS | 固定一股四天；本例因子为 1 |
| 事件三字段 Qlib 对照 | PASS | 固定历史产物的小范围数值/状态投影 |
| 真停牌、缺失/冲突故障样本 | FIXTURE UNAVAILABLE / NOT RUN | 已明确说明规则；没有编造真实样本 |
| daily 四场景、repair、resume、离线恢复 | NOT RUN：最终集成 | 展示既有入口与行为，未本次发起操作 |
| 当前产品完整测试 suite / 最终五类 View / full admission | NOT RUN | 不能引用早期 488/492 结果冒充最终代码整体验收 |
| 600069.SH 真实来源冲突最终纠正/准入 | NOT RUN / 未通过本课程验证 | 原列表与公告不一致，不能拿 synthetic 几何测试替代 |
| Research Feature/Label / UI 跨仓接入 | 未实现 / NOT RUN | Data 调用可用；跨仓构建与权限控制不能据此记 PASS |
| 截止 2026-09-24 的正式 bulk | NOT RUN，未授权启动 | 来源增量、最终规模性能和空间还欠实际证据 |

**课程完成与数据准入是不同结果。** 本次交付含 Git 中的课程源码，以及单独保存在本地 review 目录的完整执行版 Notebook 与 HTML。正式命名/协议迁移、最终全链路证据、bulk 批准与 promotion 都不属于课程写入权限。

### 17. 时间和空间：哪些数可以用，哪些不能推算？

盘点记录：整个 `/var/lib/axiom-data` 分配空间约 69.65 GB，包含 forensic；明确定义的正式 artifact 路径集合约 66.14 GB，Raw payload 清单声明逻辑约 10.95 GB。三者集合/计量方式不同，不能相加或简单相减。库存时可用空间约 620.4 GB 是时点样本，不是容量预留，更不是峰值空间预测。

| bulk 阶段 | 时间需要的实测 | 空间需要的实测 | 当前结论 |
|---|---|---|---|
| 来源补齐至 9/24 | 增量请求数、分页/限流、重试与响应大小 | 新原件、请求记录、失败保留 | 未测，NOT RUN |
| Canonical 增量/纠错 | 操作准备固定成本、受影响分区/依赖、恢复行为 | 新分区、保留旧版、staging | 最终证据缺口 |
| 五类 View | 全族证券×日期×字段形状、固定/边际成本、冷热/复用前提 | 每族新落盘字节、峰值内存/临时文件 | 最终代表测量缺口 |
| 最终验证与恢复 | exact plan/scope、闭包/投影检查成本 | 重试/恢复保留与备份空间 | 未形成终态估计 |

可信总耗时应分阶段相加，并考虑已有验证产物的可复用条件；可信峰值空间应含新增持久 bytes、同时存在的 staging、重试余量和必须保留的旧版本。短四天读取耗时不能外推全历史；旧 31.6 小时估计不能当当前结果。**这里没有可支持的 bulk 总时长或峰值 GB，保持证据缺口，不造 ETA。**

In [ ]:
saved=formal_inventory["formal_inventory"]
print("历史库存时点",formal_inventory["inventory"]["created_at_utc"])
print("Snapshot",saved["snapshot_count"],"| DomainCommit 目录与 View/Raw 详细清单见 deprecated 库存副本。")
# 当前 free disk 只用于阅读状态，不能当最终容量验收。
import shutil
free=shutil.disk_usage(formal_root).free
print(f"课程执行时可用空间 {free:,} bytes（{free/1e9:.2f} GB，时点值，非预留）。")
print("bulk 总耗时：未证实；新增持久空间与峰值 staging/retry：未证实。")

**第四段自测**：两列 close 一样，是否说明 View 完全等价？为什么 Label 可以看未来，而 Feature 不行？UI 画出四天曲线能否用于整月研究？Notebook 所有读取通过后，可以启动正式 bulk 吗？

<details><summary>对照答案</summary>

数值相等仍可能有不同价格口径、锚点、cutoff 和用途。Label 的未来区间是明确授权结果窗口，Feature 必须满足决策时可知限制；权限及 lineage 分开，当前跨仓隔离尚欠实现。UI 部分展示要明确缺口，不能借展示能力越过研究准入。课程只读通过说明这些教学调用成功，最终测试、全链路准入及成本估计未齐，bulk 仍需单独明确授权。

</details>

### 阅读后的操作入口

日常调用前，先固定四件事：**哪张版本清单、哪些证券和日期、允许知道到什么时候、缺失与证据是否满足用途**。再选真实公共读取或明确写操作计划。

- [版本模型](../docs/operations/version-model.md) 与 [磁盘布局](../docs/operations/physical-layout.md)：补查对象与持久组织。
- [View 构建](../docs/operations/materialize-views.md) 与 [完整需求准入](../docs/operations/full-admission.md)：准备可供指定用途消费的产物。
- [历史输入引用](../deprecated/data_learning/README.md) 与 [准确 fixture 身份](../deprecated/data_learning/fixture.json)：追查本课程所有业务标签对应原件与行。

Notebook 是学习和检查窗口；生产计算走已有公共入口。